In [1]:
# sample input format that comes from ML model 

#{
#     "uploaded_image": encoded_image,
#     "streetlight_count": len(boxes),
#     "on": on_count,
#     "dim": dim_count,
#     "off": off_count,
#     "details": details
# }

In [2]:
import os
import json
from openai import AsyncOpenAI
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

In [4]:
async def gen_prompt(data):
    
    prompt = f"""
        You are an AI-powered streetlight monitoring assistant.

        An uploaded streetlight image is provided together with ML detection results.
        Use both the image and the detection data to generate a professional, human-readable report.

        ML Detection Results:
        - Total Streetlights: {data["streetlight_count"]}
        - ON Lights: {data["on"]}
        - DIM Lights: {data["dim"]}
        - OFF Lights: {data["off"]}
        - Detection Details: {data["details"]}

        Instructions:
        - Analyze the uploaded image together with the ML output.
        - Describe the overall streetlight condition naturally.
        - Mention operational, dim, and faulty streetlights.
        - Highlight maintenance concerns if necessary.
        - Keep the response concise, professional, and humanized.
        - Do not mention Base64 data in the response.

        Return ONLY in this JSON format:

        {{
        "output": "your human-readable report here"
        }}
    """
    
    return prompt

In [5]:
async def gpt(prompt, base64_image):
    
    gpt_client = AsyncOpenAI(api_key=OPENAI_API_KEY)
    
    chat_prompt = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{base64_image}"}}
            ]
        }
    ]

    completion = await gpt_client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=chat_prompt,
        max_tokens=800,
        temperature=0,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0
    )

    response = completion.choices[0].message.content
    return json.loads(response)

In [6]:
async def llm_reporting(data):
    
    base64_image = data["uploaded_image"]
    prompt = await gen_prompt(data)
    response = await gpt(prompt, base64_image)
    
    return response

In [7]:
import base64

img_path = r"C:\Users\ASER\Documents\Deakin\LLM-Reporting-System\sample_imgs\sample1.jpg"

with open(img_path, "rb") as img_file:
    encoded_image = base64.b64encode(
        img_file.read()
    ).decode("utf-8")

In [8]:
data = {
    "uploaded_image": encoded_image,
    "streetlight_count": "3",
    "on": "1",
    "dim": "0",
    "off": "2",
    "details": "details"
}

response = await llm_reporting(data)
response["output"]

'The current streetlight monitoring indicates a total of three streetlights in the area. Of these, one streetlight is operational and illuminated, while two are currently off. There are no dimly lit streetlights detected. The two non-functioning streetlights may require inspection and maintenance to ensure proper lighting and safety in the vicinity. Overall, prompt attention to the off streetlights is recommended to maintain consistent street illumination.'

In [10]:
print(response)

{'output': 'The current streetlight monitoring indicates a total of three streetlights in the area. Of these, one streetlight is operational and illuminated, while two are currently off. There are no dimly lit streetlights detected. The two non-functioning streetlights may require inspection and maintenance to ensure proper lighting and safety in the vicinity. Overall, prompt attention to the off streetlights is recommended to maintain consistent street illumination.'}


In [ ]:
{'output': 'The current streetlight monitoring indicates a total of three streetlights in the area.'
' Of these, one streetlight is operational and illuminated, while two are currently off.'
' There are no dimly lit streetlights detected.'
' The two non-functioning streetlights may require inspection and maintenance to ensure proper lighting and safety in the vicinity. '
'Overall, prompt attention to the off streetlights is recommended to maintain consistent street illumination.'}